In [1]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_Token")
login(token = secret_value_0)
print(":done")


:done


In [9]:
!mkdir -p /kaggle/working/fd_llm_output
!cp -r /kaggle/input/datasets/bhavyranka/checkpoint-600-backup-zip/* /kaggle/working/fd_llm_output/

In [2]:
!pip install -U bitsandbytes peft
!install transformers==4.38.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 40.1 MB/s eta 0:00:00
  Attempting uninstall: peft
    Found existing installation: peft 0.18.1
    Uninstalling peft-0.18.1:
      Successfully uninstalled peft-0.18.1
install: missing destination file operand after 'transformers==4.38.0'
Try 'install --help' for more information.


In [3]:
import os
import json
import random
import logging
from collections import Counter
 
import numpy as np
import torch
 
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
from sklearn.model_selection import train_test_split
 
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import PeftModel


In [4]:
DATA_PATH = "/kaggle/input/datasets/bhavyranka/json-custom/cwru_stat_dataset.json"
DATA_TYPE = "stat"                        # "stat" | "fft"
 
# Base model (same one you fine-tuned from)
MODEL_ID  = "meta-llama/Meta-Llama-3-8B-Instruct"
 
# Path to your saved checkpoint — change to whichever checkpoint you want
CHECKPOINT_DIR = "/kaggle/working/fd_llm_output"
 
LOAD_IN_4BIT = True
MAX_SEQ_LEN  = 1024
SEED         = 42
TEST_SIZE    = 0.10     # same split as training so test set is identical
 
OUTPUT_DIR   = "/kaggle/working/fd_llm_eval"


In [5]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)
 
FAULT_LABELS = ["NO", "IRF", "ORF", "REF"]
LABEL2ID     = {l: i for i, l in enumerate(FAULT_LABELS)}
 
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


In [6]:
logger.info(f"Loading data from {DATA_PATH}")
with open(DATA_PATH, "r") as f:
    data = json.load(f)
logger.info(f"Total samples: {len(data)}")
logger.info(f"Label distribution [full]: {dict(Counter(d['output'] for d in data))}")
 
_, test_data = train_test_split(
    data,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=[d["output"] for d in data]
)
logger.info(f"Test samples (held-out 10%): {len(test_data)}")
logger.info(f"Label distribution [test]: {dict(Counter(d['output'] for d in test_data))}")


18:31:49 | INFO | Loading data from /kaggle/input/datasets/bhavyranka/json-custom/cwru_stat_dataset.json
18:31:49 | INFO | Total samples: 3691
18:31:49 | INFO | Label distribution [full]: {'NO': 1034, 'IRF': 885, 'ORF': 887, 'REF': 885}
18:31:49 | INFO | Test samples (held-out 10%): 370
18:31:49 | INFO | Label distribution [test]: {'REF': 89, 'IRF': 89, 'ORF': 89, 'NO': 103}


In [7]:
logger.info(f"Loading base model: {MODEL_ID}")
 
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
 
bnb_config = None
if LOAD_IN_4BIT:
    try:
        import bitsandbytes
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )
        logger.info("4-bit quantisation enabled.")
    except ImportError:
        logger.warning("bitsandbytes not found — loading in bfloat16.")
 
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16 if bnb_config is None else None,
    device_map="auto",
    trust_remote_code=True,
)
base_model.config.use_cache = False


18:31:49 | INFO | Loading base model: meta-llama/Meta-Llama-3-8B-Instruct
18:31:50 | INFO | HTTP Request: HEAD https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/main/config.json "HTTP/1.1 200 OK"
18:31:50 | INFO | HTTP Request: GET https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/main/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

18:31:50 | INFO | HTTP Request: HEAD https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
18:31:50 | INFO | HTTP Request: GET https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json: 0.00B [00:00, ?B/s]

18:31:50 | INFO | HTTP Request: HEAD https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
18:31:50 | INFO | HTTP Request: GET https://huggingface.co/api/models/meta-llama/Meta-Llama-3-8B-Instruct/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
18:31:50 | INFO | HTTP Request: GET https://huggingface.co/api/models/meta-llama/Meta-Llama-3-8B-Instruct/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
18:31:50 | INFO | HTTP Request: HEAD https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/main/tokenizer.json "HTTP/1.1 200 OK"
18:31:50 | INFO | HTTP Request: GET https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/main/tokenizer.json "HTTP/1.1 200 OK"


...

18:31:57 | INFO | HTTP Request: HEAD https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/8afb486c1db24fe5011ec46dfbe5b5dccdb575c2/model-00002-of-00004.safetensors "HTTP/1.1 302 Found"
18:31:57 | INFO | HTTP Request: HEAD https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/8afb486c1db24fe5011ec46dfbe5b5dccdb575c2/model-00003-of-00004.safetensors "HTTP/1.1 302 Found"
18:31:57 | INFO | HTTP Request: HEAD https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/8afb486c1db24fe5011ec46dfbe5b5dccdb575c2/model-00001-of-00004.safetensors "HTTP/1.1 302 Found"
18:31:57 | INFO | HTTP Request: HEAD https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/8afb486c1db24fe5011ec46dfbe5b5dccdb575c2/model-00004-of-00004.safetensors "HTTP/1.1 302 Found"
18:31:57 | INFO | HTTP Request: GET https://huggingface.co/api/models/meta-llama/Meta-Llama-3-8B-Instruct/xet-read-token/8afb486c1db24fe5011ec46dfbe5b5dccdb575c2 "HTTP/1.1 200 OK"
18:31:57 | INFO | HTTP R

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

18:33:39 | INFO | HTTP Request: HEAD https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/main/generation_config.json "HTTP/1.1 200 OK"
18:33:39 | INFO | HTTP Request: GET https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/main/generation_config.json "HTTP/1.1 200 OK"


generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

18:33:39 | INFO | HTTP Request: HEAD https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct/resolve/main/custom_generate/generate.py "HTTP/1.1 404 Not Found"


In [10]:
logger.info(f"Loading LoRA checkpoint from: {CHECKPOINT_DIR}")
model = PeftModel.from_pretrained(base_model, CHECKPOINT_DIR)
model.eval()
logger.info("Model ready for inference.")


18:35:02 | INFO | Loading LoRA checkpoint from: /kaggle/working/fd_llm_output
18:35:02 | INFO | Model ready for inference.


In [11]:
def build_inference_prompt(instruction: str, input_text: str) -> str:
    return (
        f"### Instruction:\n{instruction}\n\n"
        f"### Input:\n{input_text}\n\n"
        f"### Response:\n"
    )
 
def extract_label(generated_text: str) -> str:
    text = generated_text.strip().upper()
    for label in FAULT_LABELS:
        if label in text:
            return label
    first_word = text.split()[0] if text else "UNKNOWN"
    return first_word
 
@torch.no_grad()
def run_inference(samples, batch_size: int = 8) -> list:
    predictions = []
 
    for i in range(0, len(samples), batch_size):
        batch   = samples[i: i + batch_size]
        prompts = [build_inference_prompt(s["instruction"], s["input"]) for s in batch]
 
        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LEN,
        ).to(model.device)
 
        outputs = model.generate(
            **inputs,
            max_new_tokens=16,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
 
        # Decode only the newly generated tokens
        generated = outputs[:, inputs["input_ids"].shape[1]:]
        decoded   = tokenizer.batch_decode(generated, skip_special_tokens=True)
        predictions.extend([extract_label(d) for d in decoded])
 
        logger.info(f"  Inference: {min(i + batch_size, len(samples))}/{len(samples)}")
 
    return predictions
 
logger.info("Running inference on test set ...")
preds  = run_inference(test_data)
truths = [d["output"] for d in test_data]


18:35:06 | INFO | Running inference on test set ...
The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
18:35:28 | INFO |   Inference: 8/370
18:35:51 | INFO |   Inference: 16/370
18:36:15 | INFO |   Inference: 24/370
18:36:40 | INFO |   Inference: 32/370
18:37:04 | INFO |   Inference: 40/370
18:37:28 | INFO |   Inference: 48/370
18:37:52 | INFO |   Inference: 56/370
18:38:17 | INFO |   Inference: 64/370
18:38:41 | INFO |   Inference: 72/370
18:39:05 | INFO |   Inference: 80/370
18:39:29 | INFO |   Inference: 88/370
18:39:53 | INFO |   Inference: 96/370
18:40:17 | INFO |   Inference: 104/370
18:40:42 | INFO |   Inference: 112/370
18:41:06 | INFO |   Inference: 120/370
18:41:30 | INFO |   Inference: 128/370
18:41:54 | INFO |   Inference: 136/370
18:42:18 | INFO |   Inference: 144/370
18:42:42 | INFO |   Inference: 152/370
18:43:07 | INFO |   Inference: 160/370
18:43:31 | INFO |   Inference: 168/370
18:43:55 | INF

In [12]:
preds_clean = [p if p in FAULT_LABELS else "__INVALID__" for p in preds]
 
n_invalid = sum(1 for p in preds_clean if p == "__INVALID__")
if n_invalid:
    logger.warning(f"{n_invalid} predictions were unmappable — counted as wrong.")
 
acc  = accuracy_score(truths, preds_clean)
prec = precision_score(truths, preds_clean, labels=FAULT_LABELS, average="weighted", zero_division=0)
rec  = recall_score(truths, preds_clean, labels=FAULT_LABELS, average="weighted", zero_division=0)
f1   = f1_score(truths, preds_clean, labels=FAULT_LABELS, average="weighted", zero_division=0)
cm   = confusion_matrix(truths, preds_clean, labels=FAULT_LABELS)
 
print("\n" + "=" * 60)
print(f"  CHECKPOINT : {CHECKPOINT_DIR}")
print(f"  DATA TYPE  : {DATA_TYPE.upper()}")
print(f"  TEST SIZE  : {len(test_data)} samples")
print("=" * 60)
print(f"  Accuracy  : {acc:.4f}")
print(f"  Precision : {prec:.4f}")
print(f"  Recall    : {rec:.4f}")
print(f"  F1-Score  : {f1:.4f}")
print("\n  Classification Report:")
print(classification_report(truths, preds_clean, labels=FAULT_LABELS, zero_division=0))
print("  Confusion Matrix:")
header = "       " + "  ".join(f"{l:>5}" for l in FAULT_LABELS)
print(header)
for i, row in enumerate(cm):
    print(f"  {FAULT_LABELS[i]:>5}  {'  '.join(f'{v:5d}' for v in row)}")
print("=" * 60 + "\n")



  CHECKPOINT : /kaggle/working/fd_llm_output
  DATA TYPE  : STAT
  TEST SIZE  : 370 samples
  Accuracy  : 0.9216
  Precision : 0.9265
  Recall    : 0.9216
  F1-Score  : 0.9210

  Classification Report:
              precision    recall  f1-score   support

          NO       0.88      1.00      0.94       103
         IRF       0.89      0.96      0.92        89
         ORF       0.99      0.84      0.91        89
         REF       0.95      0.88      0.91        89

    accuracy                           0.92       370
   macro avg       0.93      0.92      0.92       370
weighted avg       0.93      0.92      0.92       370

  Confusion Matrix:
          NO    IRF    ORF    REF
     NO    103      0      0      0
    IRF      1     85      0      3
    ORF      3     10     75      1
    REF     10      0      1     78



In [13]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
 
results = {
    "checkpoint":       CHECKPOINT_DIR,
    "data_type":        DATA_TYPE,
    "test_samples":     len(test_data),
    "accuracy":         acc,
    "precision":        prec,
    "recall":           rec,
    "f1":               f1,
    "confusion_matrix": cm.tolist(),
    "invalid_preds":    n_invalid,
}
 
results_path = os.path.join(OUTPUT_DIR, f"1eval_{DATA_TYPE}_checkpoint1000.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
logger.info(f"Results saved to {results_path}")


18:53:48 | INFO | Results saved to /kaggle/working/fd_llm_eval/1eval_stat_checkpoint1000.json
